In [2]:
import numpy as np
from skglm import GeneralizedLinearEstimator
from skglm.datafits import Logistic
from skglm.penalties import SCAD
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import log_loss, roc_auc_score

In [3]:
# Generate some synthetic data
np.random.seed(42)
n_samples, n_features = 100, 20
X = np.random.randn(n_samples, n_features)
# Create some sparse true coefficients
true_coef = np.array([1.5, -2.0, 0.0, 0.8, 0.0, 0.0, -1.2, 0.0, 0.5] + [0.0] * (n_features - 9))
linear_predictor = X @ true_coef + np.random.normal(0, 0.5, n_samples)
probabilities = 1 / (1 + np.exp(-linear_predictor))
y = (probabilities > 0.5).astype(int)

In [4]:
# Define the datafit (loss function)
datafit = Logistic()

# Define the penalty (SCAD)
# alpha is the regularization strength (lambda)
# gamma is the concavity parameter, typically 3.7
scad_penalty = SCAD(alpha=0.1, gamma=3.7) # Initial alpha, will be tuned via CV

In [5]:
estimator = GeneralizedLinearEstimator(datafit=datafit, penalty=scad_penalty)
estimator.fit(X, y)

print("Fitted Coefficients (before CV):", estimator.coef_)
print("Fitted Intercept (before CV):", estimator.intercept_)

Fitted Coefficients (before CV): [[ 4.91535073 -6.97054141 -0.          0.         -0.          0.
  -4.13002504 -0.          0.          0.         -0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.        ]]
Fitted Intercept (before CV): -1.0552495837498121


In [9]:
from sklearn.datasets import make_classification
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import make_scorer, log_loss
from skglm import GeneralizedLinearEstimator
from skglm.penalties import SCAD
from skglm.datafits import Logistic
import numpy as np

# Data
X, y = make_classification(n_samples=200, n_features=50, n_informative=10, random_state=0)

# Estimator
GeneralizedLinearEstimator(
    datafit=Logistic(),
    penalty=SCAD(alpha=0.1, gamma=3.7),
    intercept_fit=True,  # ✅ Correct parameter
    max_iter=1000,
)

# Grid of top-level parameter alpha (not penalty__alpha!)
param_grid = {'alpha': np.logspace(-2, 2, 5)}

# Scorer (log_loss is a loss so use neg_log_loss to maximize it)
scorer = make_scorer(log_loss, greater_is_better=False, needs_proba=True)

# Grid Search
grid_search = GridSearchCV(
    estimator=estimator,
    param_grid=param_grid,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring=scorer,
    verbose=1
)

# Fit
grid_search.fit(X, y)

print("Best alpha:", grid_search.best_params_)
print("Best neg log-loss:", grid_search.best_score_)

TypeError: GeneralizedLinearEstimator.__init__() got an unexpected keyword argument 'intercept_fit'